In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from timm.models import create_model
from datasets import build_dataset
from compression.int_approxi_func import *

/home/chenzhiqiang/miniconda3/envs/det/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('--data-path', default='/home/chenzhiqiang/datasets/ImageNet', type=str,
                    help='dataset path')
parser.add_argument('--data-set', default='IMNET', choices=['CIFAR', 'IMNET', 'INAT', 'INAT19'],
                    type=str, help='Image Net dataset path')
parser.add_argument('--batch-size', default=64, type=int)
parser.add_argument('--num_workers', default=8, type=int)
parser.add_argument('--input-size', default=224, type=int, help='images input size')
parser.add_argument('--color-jitter', type=float, default=0.3, metavar='PCT',
                    help='Color jitter factor (default: 0.3)')
parser.add_argument('--aa', type=str, default='rand-m9-mstd0.5-inc1', metavar='NAME',
                    help='Use AutoAugment policy. "v0" or "original". " + \
                            "(default: rand-m9-mstd0.5-inc1)'),
parser.add_argument('--smoothing', type=float, default=0.1, help='Label smoothing (default: 0.1)')
parser.add_argument('--train-interpolation', type=str, default='bicubic',
                    help='Training interpolation (random, bilinear, bicubic default: "bicubic")')
parser.add_argument('--reprob', type=float, default=0.25, metavar='PCT',
                    help='Random erase prob (default: 0.25)')
parser.add_argument('--remode', type=str, default='pixel',
                    help='Random erase mode (default: "pixel")')
parser.add_argument('--recount', type=int, default=1,
                    help='Random erase count (default: 1)')
parser.add_argument('--resplit', action='store_true', default=False,
                    help='Do not random erase first (clean) augmentation split')
    
args = parser.parse_args(['--data-set', 'IMNET'])

In [ ]:
model = create_model('deit_small_patch16_224', pretrained=True)
datasets, nb_classes = build_dataset(is_train=True, args=args)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
datasets, nb_classes = build_dataset(is_train=False, args=args) # Use val set for evaluation
    
data_loader = torch.utils.data.DataLoader(
    datasets, 
    batch_size=args.batch_size,
    num_workers=args.num_workers,
    pin_memory=True,
    shuffle=True
)

data_iter = iter(data_loader)
images, targets = next(data_iter)
images = images[:32].to(device)
model.to(device)

## 1. Eval Cosine Similarity

In [ ]:
from compression.utils import ActivationCaptureHook, register_eval_hooks

hook_dict, handles = register_eval_hooks(model)

print("Running FP model forward pass to capture activations...")
with torch.no_grad():
    _ = model(images)

for handle in handles:
    handle.remove()

In [ ]:
print("\nEvaluating Cosine Similarity between FP and INT approximated modules...")
print("-" * 80)
print(f"{'Layer Name':<45} | {'Module Type':<15} | {'Cosine Sim':<10}")
print("-" * 80)

gelu_results = []
softmax_results = []
layernorm_results = []

# 5. Evaluate approximations and compute Cosine Similarity
for name, info in hook_dict.items():
    mod_type = info['type']
    
    # Instantiate corresponding Integer Approximation Module
    if mod_type == 'Softmax':
        fp_input = info['inputs'][0].to(device)
        fp_output = info['outputs'][0].to(device)
        int_mod = IntSoftmaxFixed().to(device)
        mod_name_str = "Softmax"
        result_list = softmax_results
        
    else:
        hook = info['hook']
        orig_module = info['orig_module']
        fp_input = hook.inputs[0].to(device)
        fp_output = hook.outputs[0].to(device)
        mod_name_str = mod_type.__name__
    
        if isinstance(orig_module, nn.GELU):
            int_mod = IntGeLU_LUT().to(device)
            result_list = gelu_results
            
        elif isinstance(orig_module, nn.LayerNorm):
            # For LayerNorm, we MUST copy the trained weights and biases from the original FP module
            int_mod = IntLayerNormOptimized(orig_module.normalized_shape, orig_module.eps).to(device)
            int_mod.weight.data = orig_module.weight.data.clone()
            int_mod.bias.data = orig_module.bias.data.clone()
            result_list = layernorm_results
        
    int_mod.eval()
    
    # 6. Forward pass through the Integer Module using the captured FP input
    with torch.no_grad():
        int_output = int_mod(fp_input)
        
    # 7. Calculate Cosine Similarity
    # Flatten all dimensions except the Batch dimension (dim=0) to calculate similarity per sample
    fp_flat = fp_output.view(fp_output.size(0), -1)
    int_flat = int_output.view(int_output.size(0), -1)
    
    # F.cosine_similarity returns shape [Batch_size]. We take the mean across the batch.
    cos_sim = F.cosine_similarity(fp_flat, int_flat, dim=1).mean().item()
    
    result_list.append((name, cos_sim))
    
    # print(f"{name:<45} | {mod_type.__name__:<15} | {cos_sim:.6f}")

print("\nEvaluating Cosine Similarity between FP and INT approximated modules...")

gelu_results = []
softmax_results = []
layernorm_results = []

# 5. Evaluate approximations and compute Cosine Similarity
for name, info in hook_dict.items():
    mod_type = info['type']
    

    
    # Instantiate corresponding Integer Approximation Module
    if mod_type == 'Softmax':
        fp_input = info['inputs'][0].to(device)
        fp_output = info['outputs'][0].to(device)
        int_mod = IntSoftmaxFixed().to(device)
        mod_name_str = "Softmax"
        result_list = softmax_results
        
    else:
        hook = info['hook']
        orig_module = info['orig_module']
        fp_input = hook.inputs[0].to(device)
        fp_output = hook.outputs[0].to(device)
        mod_name_str = mod_type.__name__
    
        if isinstance(orig_module, nn.GELU):
            int_mod = IntGeLU_LUT().to(device)
            result_list = gelu_results
            
        elif isinstance(orig_module, nn.LayerNorm):
            # For LayerNorm, we MUST copy the trained weights and biases from the original FP module
            int_mod = IntLayerNorm(orig_module.normalized_shape, orig_module.eps).to(device)
            int_mod.weight.data = orig_module.weight.data.clone()
            int_mod.bias.data = orig_module.bias.data.clone()
            result_list = layernorm_results
        
    int_mod.eval()
    
    # 6. Forward pass through the Integer Module using the captured FP input
    with torch.no_grad():
        int_output = int_mod(fp_input)
        
    # 7. Calculate Cosine Similarity
    # Flatten all dimensions except the Batch dimension (dim=0) to calculate similarity per sample
    fp_flat = fp_output.view(fp_output.size(0), -1)
    int_flat = int_output.view(int_output.size(0), -1)
    
    # F.cosine_similarity returns shape [Batch_size]. We take the mean across the batch.
    cos_sim = F.cosine_similarity(fp_flat, int_flat, dim=1).mean().item()
    
    result_list.append((name, cos_sim))
    
    # print(f"{name:<45} | {mod_type.__name__:<15} | {cos_sim:.6f}")

print("-" * 80)
def print_section(title, results):
    if not results:
        return
    print(f"\n{title}")
    print(f"{'Layer Name':<45} | {'Cosine Sim':<10}")
    print("-" * 60)
    sims = []
    for name, cos_sim in results:
        print(f"{name:<45} | {cos_sim:.6f}")
        sims.append(cos_sim)
    avg_sim = sum(sims) / len(sims)
    # print(f"Average: {avg_sim:.6f}")
    print(f"{'Average':<45} | {avg_sim:.6f}")
    print("-" * 60)

print_section("GELU Modules:", gelu_results)
print_section("Softmax Modules:", softmax_results)
print_section("LayerNorm Modules:", layernorm_results)

print("-" * 80)
print("Evaluation completed.")

# 2. Eval IN-1K Classification Accuracy with Non-Linear replaced

In [ ]:
from compression.int_approxi_func import IntGeLU_LUT, IntLayerNorm, IntSoftmaxFixed
from tqdm import tqdm

def eval_model_accuracy(model, dataloader, device, max_batch=None):
    model.eval()
    correct, total, batch_count = 0, 0, 0

    total_batches = len(dataloader) if max_batch is None else min(max_batch, len(dataloader))

    with torch.no_grad():
        with tqdm(total=total_batches, desc='Evaluating', unit='batch') as pbar:
            for img, targets in dataloader:
                if max_batch is not None and batch_count >= max_batch:
                    break

                img = img.to(device)
                targets = targets.to(device)

                output = model(img)
                if isinstance(output, tuple):
                    output = output[0]

                _, predicted = output.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

                batch_count += 1
                pbar.update(1)  

    return 100. * correct / total

def module_replace(model, module_type, replace_fn):
    for name, module in model.named_children():
        if isinstance(module, module_type):
            setattr(model, name, replace_fn(module))
        else:
            module_replace(module, module_type, replace_fn)

In [ ]:
import copy
print("\n [1/3] Evaluating with GELU replaced by IntGeLU")
model_gelu_replace_eval = copy.deepcopy(model)
module_replace(model_gelu_replace_eval, nn.GELU, lambda _: IntGeLU_LUT().to(device))
gelu_acc = eval_model_accuracy(model_gelu_replace_eval, data_loader, device)

gelu_acc

In [ ]:
# CHECKME 
from compression.int_approxi_func import IntLayerNormOptimized
print("\n [2/3] Evaluating with LayerNorm replaced by IntLayerNorm")
model_layernorm_replace_eval = copy.deepcopy(model)
module_replace(model_layernorm_replace_eval, nn.LayerNorm, lambda orig_ln: IntLayerNormOptimized(orig_ln.normalized_shape).to(device))

for (orig_name, orig_mod), (new_name, new_mod) in zip(model.named_modules(), model_layernorm_replace_eval.named_modules()):
    if isinstance(orig_mod, nn.LayerNorm) and isinstance(new_mod, IntLayerNormOptimized):
        new_mod.weight.data.copy_(orig_mod.weight.data)
        new_mod.bias.data.copy_(orig_mod.bias.data)

layernorm_acc = eval_model_accuracy(model_layernorm_replace_eval, data_loader, device)

layernorm_acc

In [ ]:
del model_layernorm_replace_eval
torch.cuda.empty_cache()

In [ ]:
from timm.models.vision_transformer import Attention 
print("\n [3/3] Evaluating with Softmax replaced by IntSoftmaxFixed")

class IntAttention(Attention):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.int_softmax = IntSoftmaxFixed()

    def forward(self, x, *args, **kwargs):
        # Replicate the original forward pass but replace F.softmax with int_softmax
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = self.q_norm(q)
        k = self.k_norm(k)

        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        # Replace softmax here
        attn = self.int_softmax(attn)  # Use integer softmax
        attn = self.attn_drop(attn)
        x = attn @ v

        x = x.transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

model_softmax_replace_eval = copy.deepcopy(model)

def replace_attention_with_int_attn(module):
    for name, child in module.named_children():
        if isinstance(child, Attention):
            int_attn = IntAttention(
                dim=child.qkv.in_features,
                num_heads=child.num_heads,
                qkv_bias=child.qkv.bias is not None,
                attn_drop=child.attn_drop.p,
                proj_drop=child.proj_drop.p,
                norm_layer=type(child.q_norm),
            )
            # Copy weights
            int_attn.qkv.weight.data.copy_(child.qkv.weight.data)
            if child.qkv.bias is not None:
                int_attn.qkv.bias.data.copy_(child.qkv.bias.data)
            int_attn.proj.weight.data.copy_(child.proj.weight.data)
            int_attn.proj.bias.data.copy_(child.proj.bias.data)
            # Copy norm weights if they exist (though they are Identity in this model)
            if hasattr(child.q_norm, 'weight') and child.q_norm.weight is not None:
                int_attn.q_norm.weight.data.copy_(child.q_norm.weight.data)
            if hasattr(child.k_norm, 'weight') and child.k_norm.weight is not None:
                int_attn.k_norm.weight.data.copy_(child.k_norm.weight.data)
            setattr(module, name, int_attn)
        else:
            replace_attention_with_int_attn(child)
            
replace_attention_with_int_attn(model_softmax_replace_eval)
model_softmax_replace_eval = model_softmax_replace_eval.to(device)

softmax_acc = eval_model_accuracy(model_softmax_replace_eval, data_loader, device)

softmax_acc